# Instacart PostgreSQL Data Engineering Practice

This notebook uses the full public Instacart Market Basket Analysis dataset and adds synthetic driver/delivery tables for SQL interview practice.

Download the Kaggle CSV files into `data/instacart/raw/` before running the load cells. The public Instacart dataset contains basket/order/product data, but it does not contain real drivers, stores, or delivery events. Those logistics tables are generated below for practice only.

Dataset references:

- https://www.kaggle.com/c/instacart-market-basket-analysis/data
- https://www.kaggle.com/datasets/yasserh/instacart-online-grocery-basket-analysis-dataset
- https://www.samplayle.com/instacart.html


## Setup

Expected raw files:

```text
data/instacart/raw/aisles.csv
data/instacart/raw/departments.csv
data/instacart/raw/orders.csv
data/instacart/raw/products.csv
data/instacart/raw/order_products__prior.csv
data/instacart/raw/order_products__train.csv
```

Set these environment variables before connecting to PostgreSQL: `POSTGRES_HOST`, `POSTGRES_PORT`, `POSTGRES_DB`, `POSTGRES_USER`, and `POSTGRES_PASSWORD`.


In [ ]:
# Uncomment if your notebook environment does not already have these packages.
# %pip install pandas sqlalchemy psycopg


In [ ]:
from pathlib import Path
import os
import random
from datetime import datetime, timedelta

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

RAW_DIR = Path("data/instacart/raw")
GENERATED_DIR = Path("data/instacart/generated")

TABLE_FILES = {
    "aisles": "aisles.csv",
    "departments": "departments.csv",
    "orders": "orders.csv",
    "products": "products.csv",
    "order_products_prior": "order_products__prior.csv",
    "order_products_train": "order_products__train.csv",
}

missing_files = [name for name in TABLE_FILES.values() if not (RAW_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(
        "Missing Instacart CSV files in data/instacart/raw/: "
        + ", ".join(missing_files)
    )

print("Raw data folder is ready:", RAW_DIR)


In [ ]:
db_url = URL.create(
    "postgresql+psycopg",
    username=os.getenv("POSTGRES_USER", "postgres"),
    password=os.getenv("POSTGRES_PASSWORD", ""),
    host=os.getenv("POSTGRES_HOST", "localhost"),
    port=int(os.getenv("POSTGRES_PORT", "5432")),
    database=os.getenv("POSTGRES_DB", "postgres"),
)

engine = create_engine(db_url)

with engine.begin() as conn:
    conn.execute(text("SELECT 1"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS raw_instacart"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS practice_delivery"))
    conn.execute(text("CREATE SCHEMA IF NOT EXISTS analytics"))

print("Connected and schemas are ready.")


## Load Instacart CSVs

This loader uses chunks so it can handle the larger `order_products__prior.csv` file. For production-scale loads, Postgres `COPY` is faster, but `to_sql` keeps this exercise portable inside the notebook.


In [ ]:
def normalize_columns(frame):
    frame = frame.copy()
    frame.columns = [column.strip().lower() for column in frame.columns]
    return frame


def load_csv_to_postgres(table_name, file_name, schema="raw_instacart", chunksize=100_000):
    path = RAW_DIR / file_name
    first_chunk = True
    loaded_rows = 0

    for chunk in pd.read_csv(path, chunksize=chunksize):
        chunk = normalize_columns(chunk)
        chunk.to_sql(
            table_name,
            engine,
            schema=schema,
            if_exists="replace" if first_chunk else "append",
            index=False,
            chunksize=5_000,
            method="multi",
        )
        loaded_rows += len(chunk)
        first_chunk = False

    return loaded_rows


load_counts = {}
for table_name, file_name in TABLE_FILES.items():
    load_counts[table_name] = load_csv_to_postgres(table_name, file_name)

load_counts


In [ ]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE OR REPLACE VIEW analytics.order_lines AS
        SELECT 'prior' AS source_table, order_id, product_id, add_to_cart_order, reordered
        FROM raw_instacart.order_products_prior
        UNION ALL
        SELECT 'train' AS source_table, order_id, product_id, add_to_cart_order, reordered
        FROM raw_instacart.order_products_train
    """))

    index_statements = [
        "CREATE INDEX IF NOT EXISTS idx_orders_order_id ON raw_instacart.orders(order_id)",
        "CREATE INDEX IF NOT EXISTS idx_orders_user_order_number ON raw_instacart.orders(user_id, order_number)",
        "CREATE INDEX IF NOT EXISTS idx_products_product_id ON raw_instacart.products(product_id)",
        "CREATE INDEX IF NOT EXISTS idx_order_products_prior_order_id ON raw_instacart.order_products_prior(order_id)",
        "CREATE INDEX IF NOT EXISTS idx_order_products_train_order_id ON raw_instacart.order_products_train(order_id)",
        "CREATE INDEX IF NOT EXISTS idx_order_products_prior_product_id ON raw_instacart.order_products_prior(product_id)",
        "CREATE INDEX IF NOT EXISTS idx_order_products_train_product_id ON raw_instacart.order_products_train(product_id)",
    ]
    for statement in index_statements:
        conn.execute(text(statement))

print("Analytics view and raw indexes are ready.")


## Generate Synthetic Driver and Delivery Data

The rows below are deterministic and tied to real Instacart `order_id` values. They introduce realistic interview edge cases: canceled deliveries, late deliveries, missing events, event-order mistakes, zero-tip orders, and high-volume drivers.


In [ ]:
orders_for_delivery = pd.read_sql(
    text("""
        SELECT order_id, user_id, order_number, order_dow, order_hour_of_day, eval_set
        FROM raw_instacart.orders
        WHERE eval_set IN ('prior', 'train')
        ORDER BY order_id
        LIMIT 50000
    """),
    engine,
)

if orders_for_delivery.empty:
    raise ValueError("No Instacart orders were loaded.")

stores = pd.DataFrame(
    [
        {"store_id": 1, "store_name": "North Market", "city": "Seattle", "state": "WA"},
        {"store_id": 2, "store_name": "Lakeview Foods", "city": "Seattle", "state": "WA"},
        {"store_id": 3, "store_name": "Mission Grocery", "city": "San Francisco", "state": "CA"},
        {"store_id": 4, "store_name": "SoMa Fresh", "city": "San Francisco", "state": "CA"},
        {"store_id": 5, "store_name": "River North Market", "city": "Chicago", "state": "IL"},
        {"store_id": 6, "store_name": "Logan Square Foods", "city": "Chicago", "state": "IL"},
        {"store_id": 7, "store_name": "East Austin Pantry", "city": "Austin", "state": "TX"},
        {"store_id": 8, "store_name": "South Lamar Grocery", "city": "Austin", "state": "TX"},
    ]
)

driver_rows = []
for driver_num in range(1, 49):
    home_store_id = ((driver_num - 1) % len(stores)) + 1
    driver_rows.append(
        {
            "driver_id": f"DRV{driver_num:03d}",
            "driver_name": f"Driver {driver_num:03d}",
            "home_store_id": home_store_id,
            "vehicle_type": ["bike", "sedan", "suv", "van"][driver_num % 4],
            "active": driver_num % 17 != 0,
        }
    )
drivers = pd.DataFrame(driver_rows)

base_time = datetime(2026, 1, 5, 6, 0, 0)
delivery_rows = []
event_rows = []

for row in orders_for_delivery.itertuples(index=False):
    order_id = int(row.order_id)
    user_id = int(row.user_id)
    store_id = (user_id % len(stores)) + 1
    store_drivers = drivers[drivers["home_store_id"] == store_id].reset_index(drop=True)
    driver = store_drivers.iloc[order_id % len(store_drivers)]

    accepted_at = base_time + timedelta(
        days=order_id % 120,
        hours=int(row.order_hour_of_day),
        minutes=order_id % 47,
    )
    promised_at = accepted_at + timedelta(minutes=90)
    canceled = order_id % 37 == 0
    delayed = order_id % 11 == 0
    delivery_minutes = 55 + (order_id % 42) + (45 if delayed else 0)
    delivered_at = None if canceled else accepted_at + timedelta(minutes=delivery_minutes)
    status = "canceled" if canceled else ("delayed" if delivered_at > promised_at else "delivered")

    delivery_id = f"DEL{order_id}"
    delivery_rows.append(
        {
            "delivery_id": delivery_id,
            "order_id": order_id,
            "store_id": store_id,
            "driver_id": driver.driver_id,
            "status": status,
            "accepted_at": accepted_at,
            "promised_at": promised_at,
            "delivered_at": delivered_at,
            "tip_cents": 0 if canceled or order_id % 19 == 0 else 200 + (order_id % 1800),
            "updated_at": delivered_at or accepted_at + timedelta(minutes=18),
        }
    )

    event_rows.append({"delivery_id": delivery_id, "event_sequence": 1, "event_name": "accepted", "event_ts": accepted_at})

    if canceled:
        event_rows.append({"delivery_id": delivery_id, "event_sequence": 2, "event_name": "canceled", "event_ts": accepted_at + timedelta(minutes=18)})
        if order_id % 101 == 0:
            event_rows.append({"delivery_id": delivery_id, "event_sequence": 3, "event_name": "shopping_started", "event_ts": accepted_at + timedelta(minutes=24)})
        continue

    event_rows.append({"delivery_id": delivery_id, "event_sequence": 2, "event_name": "shopping_started", "event_ts": accepted_at + timedelta(minutes=7)})
    event_rows.append({"delivery_id": delivery_id, "event_sequence": 3, "event_name": "checked_out", "event_ts": accepted_at + timedelta(minutes=38)})

    if order_id % 89 != 0:
        out_for_delivery_at = accepted_at + timedelta(minutes=52)
        delivered_event_at = delivered_at
        if order_id % 97 == 0:
            out_for_delivery_at = accepted_at + timedelta(minutes=82)
            delivered_event_at = accepted_at + timedelta(minutes=72)
        event_rows.append({"delivery_id": delivery_id, "event_sequence": 4, "event_name": "out_for_delivery", "event_ts": out_for_delivery_at})
        event_rows.append({"delivery_id": delivery_id, "event_sequence": 5, "event_name": "delivered", "event_ts": delivered_event_at})
    else:
        event_rows.append({"delivery_id": delivery_id, "event_sequence": 5, "event_name": "delivered", "event_ts": delivered_at})

deliveries = pd.DataFrame(delivery_rows)
delivery_events = pd.DataFrame(event_rows)

GENERATED_DIR.mkdir(parents=True, exist_ok=True)
stores.to_csv(GENERATED_DIR / "stores.csv", index=False)
drivers.to_csv(GENERATED_DIR / "drivers.csv", index=False)
deliveries.to_csv(GENERATED_DIR / "deliveries.csv", index=False)
delivery_events.to_csv(GENERATED_DIR / "delivery_events.csv", index=False)

for table_name, frame in {
    "stores": stores,
    "drivers": drivers,
    "deliveries": deliveries,
    "delivery_events": delivery_events,
}.items():
    frame.to_sql(table_name, engine, schema="practice_delivery", if_exists="replace", index=False)

{
    "stores": len(stores),
    "drivers": len(drivers),
    "deliveries": len(deliveries),
    "delivery_events": len(delivery_events),
}


In [ ]:
with engine.begin() as conn:
    delivery_indexes = [
        "CREATE INDEX IF NOT EXISTS idx_deliveries_order_id ON practice_delivery.deliveries(order_id)",
        "CREATE INDEX IF NOT EXISTS idx_deliveries_driver_store ON practice_delivery.deliveries(driver_id, store_id)",
        "CREATE INDEX IF NOT EXISTS idx_deliveries_accepted_at ON practice_delivery.deliveries(accepted_at)",
        "CREATE INDEX IF NOT EXISTS idx_delivery_events_delivery_id ON practice_delivery.delivery_events(delivery_id)",
        "CREATE INDEX IF NOT EXISTS idx_delivery_events_event_ts ON practice_delivery.delivery_events(event_ts)",
    ]
    for statement in delivery_indexes:
        conn.execute(text(statement))

print("Synthetic delivery indexes are ready.")


In [ ]:
def run_sql(query):
    return pd.read_sql(text(query), engine)


## Data Quality Smoke Tests

Before answering analytical questions, check the row counts and synthetic foreign-key relationships.


In [ ]:
run_sql("""
    SELECT 'orders' AS table_name, COUNT(*) AS row_count FROM raw_instacart.orders
    UNION ALL
    SELECT 'products', COUNT(*) FROM raw_instacart.products
    UNION ALL
    SELECT 'order_lines', COUNT(*) FROM analytics.order_lines
    UNION ALL
    SELECT 'deliveries', COUNT(*) FROM practice_delivery.deliveries
    UNION ALL
    SELECT 'delivery_events', COUNT(*) FROM practice_delivery.delivery_events
    ORDER BY table_name
""")


In [ ]:
run_sql("""
    SELECT 'deliveries_without_order' AS check_name, COUNT(*) AS bad_rows
    FROM practice_delivery.deliveries d
    LEFT JOIN raw_instacart.orders o USING (order_id)
    WHERE o.order_id IS NULL
    UNION ALL
    SELECT 'events_without_delivery', COUNT(*)
    FROM practice_delivery.delivery_events e
    LEFT JOIN practice_delivery.deliveries d USING (delivery_id)
    WHERE d.delivery_id IS NULL
    UNION ALL
    SELECT 'duplicate_delivery_ids', COUNT(*)
    FROM (
        SELECT delivery_id
        FROM practice_delivery.deliveries
        GROUP BY delivery_id
        HAVING COUNT(*) > 1
    ) duplicates
    UNION ALL
    SELECT 'impossible_delivery_duration', COUNT(*)
    FROM practice_delivery.deliveries
    WHERE delivered_at < accepted_at
""")


## SQL Practice Questions and Solutions

Try writing each query before opening the solution cell below it.


### 1. Product Reorder Rate

Find the products ordered at least 100 times with the highest reorder rate.


In [ ]:
run_sql("""
    SELECT
        p.product_id,
        p.product_name,
        COUNT(*) AS times_ordered,
        ROUND(AVG(l.reordered::numeric), 4) AS reorder_rate
    FROM analytics.order_lines l
    JOIN raw_instacart.products p USING (product_id)
    GROUP BY p.product_id, p.product_name
    HAVING COUNT(*) >= 100
    ORDER BY reorder_rate DESC, times_ordered DESC
    LIMIT 20
""")


### 2. Department and Aisle Performance

Rank department and aisle combinations by order volume and reorder rate.


In [ ]:
run_sql("""
    SELECT
        d.department,
        a.aisle,
        COUNT(*) AS line_count,
        COUNT(DISTINCT l.order_id) AS order_count,
        ROUND(AVG(l.reordered::numeric), 4) AS reorder_rate
    FROM analytics.order_lines l
    JOIN raw_instacart.products p USING (product_id)
    JOIN raw_instacart.aisles a USING (aisle_id)
    JOIN raw_instacart.departments d USING (department_id)
    GROUP BY d.department, a.aisle
    ORDER BY order_count DESC, reorder_rate DESC
    LIMIT 25
""")


### 3. Common Product Pairs

On a sample of prior orders, find product pairs that appear together most often.


In [ ]:
run_sql("""
    WITH sample_orders AS (
        SELECT order_id
        FROM raw_instacart.orders
        WHERE eval_set = 'prior'
        ORDER BY order_id
        LIMIT 10000
    ),
    sampled_lines AS (
        SELECT op.order_id, op.product_id
        FROM raw_instacart.order_products_prior op
        JOIN sample_orders so USING (order_id)
    ),
    pairs AS (
        SELECT
            l1.product_id AS product_a,
            l2.product_id AS product_b,
            COUNT(*) AS pair_orders
        FROM sampled_lines l1
        JOIN sampled_lines l2
          ON l1.order_id = l2.order_id
         AND l1.product_id < l2.product_id
        GROUP BY l1.product_id, l2.product_id
    )
    SELECT
        pa.product_name AS product_a,
        pb.product_name AS product_b,
        pairs.pair_orders
    FROM pairs
    JOIN raw_instacart.products pa ON pa.product_id = pairs.product_a
    JOIN raw_instacart.products pb ON pb.product_id = pairs.product_b
    ORDER BY pairs.pair_orders DESC
    LIMIT 20
""")


### 4. Latest Basket Versus Customer History

Find users whose latest basket size is much larger than their previous average basket size.


In [ ]:
run_sql("""
    WITH order_sizes AS (
        SELECT order_id, COUNT(*) AS item_count
        FROM analytics.order_lines
        GROUP BY order_id
    ),
    ranked AS (
        SELECT
            o.user_id,
            o.order_id,
            o.order_number,
            s.item_count,
            ROW_NUMBER() OVER (
                PARTITION BY o.user_id
                ORDER BY o.order_number DESC
            ) AS recency_rank
        FROM raw_instacart.orders o
        JOIN order_sizes s USING (order_id)
    ),
    user_baskets AS (
        SELECT
            user_id,
            MAX(item_count) FILTER (WHERE recency_rank = 1) AS latest_item_count,
            AVG(item_count) FILTER (WHERE recency_rank > 1) AS historical_avg_item_count
        FROM ranked
        GROUP BY user_id
    )
    SELECT
        user_id,
        latest_item_count,
        ROUND(historical_avg_item_count::numeric, 2) AS historical_avg_item_count,
        ROUND((latest_item_count / NULLIF(historical_avg_item_count, 0))::numeric, 2) AS latest_to_history_ratio
    FROM user_baskets
    WHERE historical_avg_item_count IS NOT NULL
      AND latest_item_count >= historical_avg_item_count * 2
    ORDER BY latest_to_history_ratio DESC, latest_item_count DESC
    LIMIT 25
""")


### 5. On-Time Delivery Rate by Driver and Store

Find driver/store combinations with enough completed deliveries to evaluate on-time performance.


In [ ]:
run_sql("""
    SELECT
        d.driver_id,
        dr.driver_name,
        d.store_id,
        s.store_name,
        COUNT(*) AS completed_deliveries,
        ROUND(AVG((d.delivered_at <= d.promised_at)::int)::numeric, 3) AS on_time_rate,
        ROUND(AVG(EXTRACT(EPOCH FROM (d.delivered_at - d.accepted_at)) / 60)::numeric, 1) AS avg_delivery_minutes
    FROM practice_delivery.deliveries d
    JOIN practice_delivery.drivers dr USING (driver_id)
    JOIN practice_delivery.stores s USING (store_id)
    WHERE d.status IN ('delivered', 'delayed')
    GROUP BY d.driver_id, dr.driver_name, d.store_id, s.store_name
    HAVING COUNT(*) >= 20
    ORDER BY on_time_rate ASC, completed_deliveries DESC
    LIMIT 25
""")


### 6. Rolling 7-Day Delivery Volume

Compute daily delivery count and rolling 7-day delivery count for each store.


In [ ]:
run_sql("""
    WITH daily AS (
        SELECT
            store_id,
            DATE_TRUNC('day', accepted_at)::date AS delivery_day,
            COUNT(*) AS delivery_count
        FROM practice_delivery.deliveries
        GROUP BY store_id, DATE_TRUNC('day', accepted_at)::date
    )
    SELECT
        store_id,
        delivery_day,
        delivery_count,
        SUM(delivery_count) OVER (
            PARTITION BY store_id
            ORDER BY delivery_day
            ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
        ) AS rolling_7_day_delivery_count
    FROM daily
    ORDER BY store_id, delivery_day
    LIMIT 80
""")


### 7. Invalid Delivery Event Ordering

Find delivery events where the status moves backward or where work happens after a terminal event.


In [ ]:
run_sql("""
    WITH ordered_events AS (
        SELECT
            delivery_id,
            event_sequence,
            event_name,
            event_ts,
            CASE event_name
                WHEN 'accepted' THEN 1
                WHEN 'shopping_started' THEN 2
                WHEN 'checked_out' THEN 3
                WHEN 'out_for_delivery' THEN 4
                WHEN 'delivered' THEN 5
                WHEN 'canceled' THEN 5
            END AS event_rank
        FROM practice_delivery.delivery_events
    ),
    compared AS (
        SELECT
            *,
            LAG(event_name) OVER (PARTITION BY delivery_id ORDER BY event_ts, event_sequence) AS previous_event,
            LAG(event_rank) OVER (PARTITION BY delivery_id ORDER BY event_ts, event_sequence) AS previous_rank
        FROM ordered_events
    )
    SELECT
        delivery_id,
        previous_event,
        event_name,
        event_ts,
        CASE
            WHEN previous_event IN ('delivered', 'canceled') THEN 'event_after_terminal_status'
            WHEN event_rank < previous_rank THEN 'status_moved_backward'
        END AS anomaly_reason
    FROM compared
    WHERE previous_event IN ('delivered', 'canceled')
       OR event_rank < previous_rank
    ORDER BY delivery_id, event_ts
    LIMIT 50
""")


### 8. Incremental Load Pattern

Pretend the delivery table is loaded incrementally. Pull records changed after a watermark and count them by status.


In [ ]:
run_sql("""
    WITH watermark AS (
        SELECT MAX(updated_at) - INTERVAL '3 days' AS last_successful_load_at
        FROM practice_delivery.deliveries
    ),
    changed_rows AS (
        SELECT d.*
        FROM practice_delivery.deliveries d
        CROSS JOIN watermark w
        WHERE d.updated_at > w.last_successful_load_at
    )
    SELECT
        status,
        COUNT(*) AS changed_rows,
        MIN(updated_at) AS first_changed_at,
        MAX(updated_at) AS last_changed_at
    FROM changed_rows
    GROUP BY status
    ORDER BY changed_rows DESC
""")


### 9. Index Choices

For interview discussion: these indexes support the common join and filter paths used above. Use `EXPLAIN ANALYZE` on the analytical queries before and after adding them to compare plans.


In [ ]:
run_sql("""
    SELECT
        schemaname,
        tablename,
        indexname,
        indexdef
    FROM pg_indexes
    WHERE schemaname IN ('raw_instacart', 'practice_delivery')
    ORDER BY schemaname, tablename, indexname
""")
